# Phase 2: Exploratory Audio Forensics & Dataset Analysis

**Project:** AI-Based Voice Analysis for Detecting Synthetic and Deepfake Audio in Criminal Investigations

This notebook demonstrates:
1. Dataset integrity, chain of custody, and cryptographic SHA-256 evidence hashing.
2. Standardisation to mono and 16 kHz without modifying original evidence files.
3. Forensic feature extraction across 9 acoustic dimensions:
   - 20 Mel-Frequency Cepstral Coefficients (MFCCs)
   - Spectral centroid
   - Spectral bandwidth
   - Spectral rolloff
   - Zero-crossing rate
   - RMS energy
   - Chroma (12 pitch classes)
   - Spectral contrast (octave sub-bands)
   - Spectral flatness

In [ ]:
import json
from pathlib import Path

manifest_path = Path("../dataset/evidence_manifest.json")
if not manifest_path.is_file():
    manifest_path = Path("dataset/evidence_manifest.json")

if manifest_path.is_file():
    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(f"Total Evidence Files: {manifest['total_recordings_processed']}")
    print(f"Genuine Audio Files: {manifest['total_genuine_recordings']}")
    print(f"Synthetic Deepfakes: {manifest['total_synthetic_recordings']}")
    print(f"Class Distribution: {manifest['class_distribution']}")
    print(f"Sample Rates: {manifest['sample_rate_distribution']}")
    print(f"Total Duration: {manifest['audio_duration_statistics']['total_hours']:.2f} hours")
else:
    print("Manifest not found. Run 'python main.py --preprocess' to generate.")

## 2. Chain of Custody & Evidence Verification

In criminal forensic investigations, evidence integrity must be strictly maintained. Original evidence files are read-only exhibits, hashed with SHA-256 upon intake. Preprocessed audio derivatives are stored separately in `dataset/preprocessed/` with their own cryptographic audit trail.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.audio_preprocessing import compute_file_hash, get_audio_metadata

sample_real = Path("../dataset/real/linus-original-DEMO.mp3")
if not sample_real.is_file():
    sample_real = Path("dataset/real/linus-original-DEMO.mp3")

if sample_real.is_file():
    meta = get_audio_metadata(sample_real)
    print(f"Evidence File: {meta['file_name']}")
    print(f"Format: {meta['format']}, Channels: {meta['channels']}, Sample Rate: {meta['sample_rate']} Hz")
    print(f"Duration: {meta['duration_seconds']:.2f} seconds")
    print(f"SHA-256 Hash: {meta['sha256']}")
else:
    print("Sample audio not found.")

## 3. Forensic Acoustic Feature Extraction

Extract 87 standard acoustic and spectral statistics from genuine and synthetic speech signals without external dependencies or data fabrication.

In [ ]:
from src.feature_extraction import extract_features_from_file

if sample_real.is_file():
    features = extract_features_from_file(sample_real, sample_rate=16_000, n_mfcc=20)
    print(f"Extracted {len(features)} feature values for {sample_real.name}:")
    key_features = [
        'duration_seconds', 'rms_mean', 'zero_crossing_rate_mean',
        'spectral_centroid_mean', 'spectral_bandwidth_mean', 'spectral_rolloff_mean',
        'spectral_flatness_mean', 'spectral_contrast_mean', 'chroma_mean',
        'mfcc_1_mean', 'mfcc_20_mean'
    ]
    for k in key_features:
        print(f"  {k:30s}: {features[k]:.4f}")
else:
    print("Sample audio not found.")